# Week 4 - Day 2

# Unsupervised Learning: K-Means Clustering

In Week 4, Day 1, we used **Supervised Learning** to train a model with known answers.

Today, we move to **Unsupervised Learning**, where the model has no target answer and instead looks for patterns or groups in the data.


## Learning Objectives

By the end of this lesson, you should be able to:

- Explain supervised vs. unsupervised learning.
- Explain what clustering means.
- Explain how K-Means works.
- Understand `K`, centroids, distance, and cluster labels.
- Prepare data for K-Means.
- Apply feature scaling and understand why it matters.
- Use K-Means with at least 3 clusters.
- Visualize and interpret clusters.
- Use the Elbow Method and Silhouette Score to help choose `K`.
- Compare different feature choices and different values of `K`.
- Explain the limitations of K-Means.


## Why This Lesson Matters

Imagine a business has **10,000 customers**.

It knows their age, income, and spending habits, but nobody has manually labelled them as "low spender", "medium spender", or "high spender".

Instead of telling the model what the groups are, we can ask it to **discover groups with similar behaviour**.

That is the idea behind clustering.


## Recap: Supervised Learning

In supervised learning, we have:

**Features → Known Target**

For example:

`House Area + Rooms → House Price`

The model learns from examples where the correct answer is already known.


## Today's Difference

In unsupervised learning, we usually have:

**Features → No Known Target**

For example:

`Income + Spending Score → ?`

There is no "correct cluster" column telling us the answer.

The model searches for structure by itself.


## What is Unsupervised Learning?

**Unsupervised Learning is a type of Machine Learning where a model finds patterns, structures, or groups in data without a known target variable.**

Common unsupervised learning tasks include:

- Clustering
- Dimensionality reduction
- Association analysis


## What is Clustering?

**Clustering is the process of grouping similar observations together while keeping less-similar observations in different groups.**

A customer with similar income and spending behaviour should ideally be placed near customers with similar behaviour.


## Real-Life Scenario

A supermarket has customer records but no customer categories.

After clustering, the business might discover groups such as:

- Lower-income, lower-spending customers
- Middle-income, regular-spending customers
- Higher-income, higher-spending customers

The algorithm discovers the groups. **We interpret what the groups mean.**


## Other Examples of Clustering

Clustering can also be used for:

| Area | Possible Grouping |
|---|---|
| Education | Students with similar learning behaviour |
| Banking | Customers with similar transaction patterns |
| Marketing | Customers with similar purchasing behaviour |
| Healthcare | Patients with similar measured characteristics |
| Products | Products with similar price and sales patterns |


## What is K-Means?

**K-Means is a clustering algorithm that divides observations into a chosen number of groups, called clusters, based on similarity.**

The `K` represents the number of clusters we want the algorithm to create.


## What Does K Mean?

If:

- `K = 2` → create 2 clusters
- `K = 3` → create 3 clusters
- `K = 5` → create 5 clusters

The value of `K` is something we must choose carefully.


## What is a Centroid?

**A centroid is the center point of a cluster.**

Think of it as the representative center of a group.

K-Means repeatedly moves these centers until the observations are assigned to the most suitable groups.


## How K-Means Works

At a high level:

1. Choose `K`.
2. Start with `K` centroids.
3. Assign each observation to its nearest centroid.
4. Recalculate each centroid.
5. Repeat the assignment and recalculation.
6. Stop when the centroids stop changing significantly.


## Step 1: Choose K

Suppose we decide:

`K = 3`

K-Means will try to divide our observations into three groups.


## Step 2: Start with Centroids

The algorithm starts with three initial center points.

The exact starting positions can affect the path the algorithm takes, which is one reason we use settings such as `random_state` and `n_init`.


## Step 3: Assign Points

Each observation is compared with the centroids.

The observation is assigned to the closest centroid.


## Step 4: Recalculate the Centroids

After observations have been assigned, K-Means calculates a new center for each group.

This new center is based on the observations currently assigned to that cluster.


## Step 5: Repeat

The algorithm keeps doing:

**Assign → Recalculate → Assign → Recalculate**

until the clusters become stable.


## Why Distance Matters

K-Means is distance-based.

If two customers have similar values for the features we selected, they will generally be closer together in feature space.


## A Simple Distance Example

Imagine two customers:

- Customer A: income = 40, spending = 60
- Customer B: income = 42, spending = 62

Their values are close, so their distance is small.


In [1]:
import math

customer_a = (40, 60)
customer_b = (42, 62)

distance = math.sqrt(
    (customer_a[0] - customer_b[0])**2 +
    (customer_a[1] - customer_b[1])**2
)

print("Distance:", distance)


Distance: 2.8284271247461903


## Why We Start With a Small Example

The code above is not K-Means yet.

It simply helps us understand the **distance idea** that K-Means uses when deciding which observations are similar.


## A Very Important Point

K-Means does not understand words like:

- "Premium customer"
- "Poor student"
- "High spender"

It only sees the numerical features we provide.

**We give the clusters meaning after the model has created them.**


## Warm-Up: Import the Libraries

We will use:

- **Pandas** for the dataset.
- **NumPy** for numerical operations.
- **Matplotlib** for visualization.
- **Scikit-learn** for K-Means, scaling, and evaluation.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# from sklearn.cluster import KMeans
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import silhouette_score

print("Libraries successfully imported!")


Libraries successfully imported!


## Our Main Dataset: `customers.csv`

The curriculum uses a customer dataset with features such as income, spending, and age.

To make the notebook reproducible, we will create a small practice `customers.csv` first, then load it exactly as we would load a real CSV supplied by a business.


## Create the Customer Dataset

Imagine these records came from a company's customer database.

We will use:

- `Customer_ID`
- `Age`
- `Income`
- `Spending_Score`

For the first model, we will deliberately focus on **Income** and **Spending_Score** so that students can clearly see the clusters on a 2D chart.


In [3]:
customer_data = {
    "Customer_ID": [f"C{i:03d}" for i in range(1, 31)],
    "Age": [
        22, 25, 23, 27, 24,
        45, 48, 50, 46, 52,
        31, 34, 29, 36, 32,
        41, 44, 39, 47, 42,
        21, 26, 28, 23, 30,
        55, 58, 53, 60, 57
    ],
    "Income": [
        25, 30, 28, 32, 27,
        70, 75, 80, 68, 85,
        45, 50, 42, 55, 48,
        60, 65, 58, 72, 63,
        22, 35, 38, 26, 40,
        90, 95, 88, 100, 92
    ],
    "Spending_Score": [
        82, 78, 85, 80, 88,
        35, 30, 28, 40, 25,
        65, 70, 62, 72, 68,
        45, 50, 42, 38, 48,
        90, 85, 88, 92, 80,
        20, 18, 25, 15, 22
    ]
}

customers = pd.DataFrame(customer_data)

customers.to_csv("customers.csv", index=False)

print("customers.csv created successfully.")


customers.csv created successfully.


## Load the CSV

In a real project, we normally receive the dataset rather than manually creating it.

The workflow is therefore:

**CSV → Pandas DataFrame → Exploration → Preparation → Model**


In [3]:
customers = pd.read_csv("customers.csv")

customers.head()


NameError: name 'pd' is not defined

## Inspect the Dataset

Before applying Machine Learning, we should understand what is inside the dataset.

This is the same habit we developed in previous weeks.


In [ ]:
customers.info()


## Check for Missing Values

Missing values can affect Machine Learning algorithms.

Even though our practice dataset is clean, we should still check.


In [ ]:
customers.isnull().sum()


## Basic Statistics

`describe()` gives us a quick view of numerical columns such as their average, minimum, and maximum values.


In [ ]:
customers.describe()


## Visualize the Customers Before Clustering

Before building a model, we should look at the data.

A scatter plot is useful here because we have two numerical variables and want to see their relationship.


In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    customers["Income"],
    customers["Spending_Score"]
)

plt.title("Customers Before Clustering")
plt.xlabel("Income")
plt.ylabel("Spending Score")
plt.show()


## What Should We Look For?

Look for areas where points appear naturally close together.

This does **not** prove that the groups are correct. It simply gives us an initial understanding of the data before modeling.


## Choosing Our Features

For our first customer model, we choose:

- `Income`
- `Spending_Score`

Why?

Because our business question is:

> **Can we discover groups of customers based on their financial capacity and spending behaviour?**


In [ ]:
X = customers[["Income", "Spending_Score"]]

X.head()


## Why Not Use `Customer_ID`?

`Customer_ID` identifies the customer.

It does not describe customer behaviour.

Using an ID as a Machine Learning feature would introduce meaningless information into the model.


## What About Age?

Age could be useful, but including it changes the question.

If we use:

`Age + Income + Spending Score`

we are asking the model to group customers using all three characteristics.

For our first example, we keep only two features because the clusters are easier to visualize and understand.


# Part 1 — Our First K-Means Model

The curriculum requires us to apply K-Means with at least **3 clusters**.

We will first build the simplest version so we can understand the workflow before improving it.


## Why Start With a Simple Model?

Beginners should first understand:

**Data → Features → K-Means → Cluster Labels → Visualization**

After that, we will introduce scaling, better K selection, and evaluation.

This keeps the reasoning clear instead of hiding everything inside one large code cell.


## Train K-Means With 3 Clusters

We use `n_clusters=3` because we want three customer groups for this first demonstration.


In [ ]:
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(X)

clusters[:10]


## What Did `fit_predict()` Do?

`fit_predict()` performs two jobs:

1. **Fit** — learns the cluster structure from the data.
2. **Predict** — returns the cluster label assigned to each observation.

The result is a number such as `0`, `1`, or `2`.


## Add the Cluster Labels to the Dataset

We store the result so we can inspect and visualize the groups.


In [ ]:
customers["Cluster"] = clusters

customers.head()


## Count Customers in Each Cluster

Before interpreting a cluster, it is useful to know how many observations belong to it.


In [ ]:
customers["Cluster"].value_counts().sort_index()


## Visualize the First Model

Now we colour the points according to the cluster assigned by K-Means.


In [ ]:
plt.figure(figsize=(9, 6))

plt.scatter(
    customers["Income"],
    customers["Spending_Score"],
    c=customers["Cluster"]
)

plt.xlabel("Income")
plt.ylabel("Spending Score")
plt.title("Customer Segmentation with K-Means")
plt.show()


## Important: What Do `0`, `1`, and `2` Mean?

They are simply cluster labels.

For example:

`Cluster 0` does **not** automatically mean "low-value".

We must inspect the characteristics of each group before naming it.


## Find the Centroids

K-Means stores its learned center points in:

`cluster_centers_`


In [ ]:
kmeans.cluster_centers_


## Why Are the Centroids Scaled-Looking?

In this first model we trained directly on the original features.

Therefore, the centroid values are still expressed in the same units as:

- Income
- Spending Score


In [ ]:
centers = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=["Income", "Spending_Score"]
)

centers


## Plot the Centroids

The centroid represents the approximate center of each cluster.


In [ ]:
plt.figure(figsize=(9, 6))

plt.scatter(
    customers["Income"],
    customers["Spending_Score"],
    c=customers["Cluster"],
    alpha=0.7
)

plt.scatter(
    centers["Income"],
    centers["Spending_Score"],
    marker="X",
    s=200
)

plt.xlabel("Income")
plt.ylabel("Spending Score")
plt.title("Clusters and Their Centroids")
plt.show()


## Profile the Clusters

Instead of guessing what a cluster means from its number, calculate its average characteristics.

This is how we turn model output into an interpretable business result.


In [ ]:
cluster_profile = customers.groupby("Cluster")[
    ["Age", "Income", "Spending_Score"]
].mean().round(2)

cluster_profile


## Interpreting the Profile

Suppose a cluster has:

- High average income
- High average spending

A reasonable business interpretation could be:

**High-value customers**

Another cluster may have lower income and lower spending.

We give the cluster a name based on its **data**, not its numerical label.


# Part 2 — Why Feature Scaling Matters

Our first model worked, but there is an important issue.

K-Means uses distance.

That means the numerical scale of our features can affect the result.


## A Simple Example

Suppose we have:

- Age: 18–70
- Income: 20–100,000
- Spending Score: 1–100

Income is measured on a much larger numerical scale.

Without scaling, a distance-based algorithm can give too much influence to the feature with the larger numerical values.


## What is Feature Scaling?

**Feature scaling is the process of transforming numerical features so that their values are on comparable scales.**

This is especially important for algorithms that depend on distance.


## Two Common Scaling Approaches

| Method | Main Idea |
|---|---|
| Min-Max Scaling | Rescales values to a chosen range, commonly 0–1 |
| Standardization | Centers values around 0 and scales according to standard deviation |

Both can be useful.

For this lesson, we choose **StandardScaler**.


## Why Choose StandardScaler Here?

K-Means works with distances.

Standardization is a common choice when numerical variables have different units or ranges because it puts the features on a comparable scale while preserving their relative variation.

We already introduced normalization and standardization in Week 3, so this is where that preprocessing concept becomes useful in Machine Learning.


In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled[:5]


## What Did `fit_transform()` Do?

`fit` learns the scaling information from the data.

`transform` applies that scaling.

`fit_transform()` performs both operations in one step.


## Check the Scaled Data

The values may now look very different from the original income and spending values.

That is expected.

The model uses the scaled values, while we can still interpret our final results using the original data.


In [ ]:
print("Original values:")
print(X.head())

print("\nScaled values:")
print(X_scaled[:5])


## Compare K-Means Without Scaling

We already trained one model without scaling.

We will keep it as a useful comparison rather than pretending it never happened.


In [ ]:
unscaled_model = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

unscaled_labels = unscaled_model.fit_predict(X)

unscaled_score = silhouette_score(X, unscaled_labels)

print("Unscaled Silhouette Score:", unscaled_score)


## Train K-Means With Scaling

Now we use the standardized features.


In [ ]:
scaled_model = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

scaled_labels = scaled_model.fit_predict(X_scaled)

scaled_score = silhouette_score(X_scaled, scaled_labels)

print("Scaled Silhouette Score:", scaled_score)


## Why Are We Comparing Them?

The purpose is not to blindly say "scaled is always better".

The point is to understand that:

**preprocessing choices can affect a Machine Learning model.**

For distance-based algorithms, scaling is usually a very important consideration.


## Use the Scaled Model as Our Main Model

For the rest of this notebook, we will use the scaled version because it is the more appropriate workflow when feature ranges differ.


In [ ]:
customers["Cluster"] = scaled_labels

customers.head()


# Part 3 — How Do We Choose K?

We have used `K = 3`, but we should not always choose 3 simply because it is convenient.

In a real project, we need a method for comparing possible values of `K`.


## Method 1: Elbow Method

**The Elbow Method compares the model's inertia across different values of K.**

We look for a point where increasing K starts producing much smaller improvements.


## What is Inertia?

**Inertia measures the total squared distance between observations and the centroids of their assigned clusters.**

Lower inertia means the observations are closer to their cluster centers.

However, inertia almost always decreases as K increases, so we do not simply choose the largest K.


## Test Several K Values

We will test values from 2 to 8.

We start at 2 because a single cluster does not perform meaningful segmentation.


In [ ]:
inertia_values = []

for k in range(2, 9):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(X_scaled)
    inertia_values.append(model.inertia_)

inertia_values


## Plot the Elbow Curve

The curve helps us see where the improvement begins to slow down.


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    range(2, 9),
    inertia_values,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()


## How Do We Read the Elbow?

Imagine the curve drops sharply from:

`K=2 → K=3`

but then the improvements become much smaller.

The bend around `K=3` may be a reasonable candidate.

The elbow is a **guide**, not a magic answer.


## Method 2: Silhouette Score

**The Silhouette Score measures how well observations fit inside their own cluster compared with neighbouring clusters.**

It considers both:

- Cohesion: how close an observation is to its own cluster.
- Separation: how far it is from other clusters.


## Silhouette Score Range

The score is generally between **-1 and 1**.

- A higher score generally indicates better-separated clusters.
- Around 0 suggests overlapping clusters.
- A negative score can indicate poorly assigned observations.

We use the score as evidence, not as the only decision-maker.


In [ ]:
silhouette_values = []

for k in range(2, 9):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X_scaled)

    score = silhouette_score(
        X_scaled,
        labels
    )

    silhouette_values.append(score)

silhouette_values


## Plot the Silhouette Scores


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    range(2, 9),
    silhouette_values,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score by K")
plt.show()


## Elbow vs Silhouette

The two methods answer slightly different questions:

| Method | Main Question |
|---|---|
| Elbow Method | How much does adding another cluster improve compactness? |
| Silhouette Score | How well separated and cohesive are the clusters? |

When possible, use both alongside visualization and domain knowledge.


## Why We Keep K=3 for the Main Example

The curriculum specifically asks us to apply K-Means with **at least 3 clusters**.

We therefore keep `K=3` as our main teaching example while showing students how a real analyst would investigate whether another K may be better.


# Part 4 — Final K=3 Customer Model

Now we know the full workflow:

**Inspect → Select Features → Scale → Choose K → Train → Visualize → Profile → Evaluate**


In [ ]:
final_model = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

customers["Cluster"] = final_model.fit_predict(X_scaled)

customers.head()


## Final Silhouette Score

We now evaluate the final clustering arrangement using the Silhouette Score.


In [ ]:
final_score = silhouette_score(
    X_scaled,
    customers["Cluster"]
)

print("Final Silhouette Score:", round(final_score, 3))


## Visualize the Final Clusters

This is the main visualization required by the curriculum.


In [ ]:
plt.figure(figsize=(9, 6))

plt.scatter(
    customers["Income"],
    customers["Spending_Score"],
    c=customers["Cluster"],
    alpha=0.75
)

plt.xlabel("Income")
plt.ylabel("Spending Score")
plt.title("Final Customer Segmentation")
plt.show()


## Visualize the Final Centroids

Because the model learned the centroids in scaled space, we convert them back to the original feature scale before plotting and interpreting them.


In [ ]:
final_centers_scaled = final_model.cluster_centers_

final_centers_original = scaler.inverse_transform(
    final_centers_scaled
)

final_centers = pd.DataFrame(
    final_centers_original,
    columns=["Income", "Spending_Score"]
)

final_centers


In [ ]:
plt.figure(figsize=(9, 6))

plt.scatter(
    customers["Income"],
    customers["Spending_Score"],
    c=customers["Cluster"],
    alpha=0.7
)

plt.scatter(
    final_centers["Income"],
    final_centers["Spending_Score"],
    marker="X",
    s=250
)

plt.xlabel("Income")
plt.ylabel("Spending Score")
plt.title("Customer Clusters and Centroids")
plt.show()


## Build a Cluster Profile

We now summarize each group using its average age, income, spending score, and number of customers.


In [ ]:
final_profile = customers.groupby("Cluster").agg(
    Average_Age=("Age", "mean"),
    Average_Income=("Income", "mean"),
    Average_Spending_Score=("Spending_Score", "mean"),
    Customer_Count=("Customer_ID", "count")
).round(2)

final_profile


## Why Use a Profile Table?

The scatter plot shows us **where** the groups are.

The profile table tells us **what characterizes each group**.

Both are needed for interpretation.


## Naming the Clusters

Do not assume:

`Cluster 0 = Low`

or:

`Cluster 1 = Premium`

Cluster numbers are arbitrary.

Instead, inspect the averages and then create meaningful descriptions.


In [ ]:
profile_sorted = final_profile.sort_values(
    "Average_Spending_Score",
    ascending=False
)

profile_sorted


## Example Business Interpretation

If one group has high income and high spending, a business might describe it as:

**High-value / high-spending customers**

If another has lower income and lower spending:

**Lower-income / lower-spending customers**

The exact labels should come from the actual profile values.


## A Cluster Is Not a Personality Test

K-Means only sees the features we give it.

If we cluster customers using income and spending, the resulting groups describe patterns in those variables.

We should not claim the model discovered things such as personality, loyalty, or motivation unless those were actually measured.


# Part 5 — Different Feature Choices Produce Different Groups

This is an important Machine Learning lesson.

The model does not decide the question for us.

**Our feature selection helps define the question.**


## Example 1: Age + Spending Score

Suppose the business is interested in:

> "How does age relate to spending behaviour?"

Then `Age` and `Spending_Score` make more sense than `Income` and `Spending_Score`.


In [ ]:
X_age_spend = customers[
    ["Age", "Spending_Score"]
]

age_spend_scaler = StandardScaler()

X_age_spend_scaled = age_spend_scaler.fit_transform(
    X_age_spend
)

X_age_spend_scaled[:5]


In [ ]:
age_spend_model = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

age_spend_labels = age_spend_model.fit_predict(
    X_age_spend_scaled
)

customers["Age_Spend_Cluster"] = age_spend_labels

customers.head()


In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    customers["Age"],
    customers["Spending_Score"],
    c=customers["Age_Spend_Cluster"]
)

plt.xlabel("Age")
plt.ylabel("Spending Score")
plt.title("Age and Spending Clusters")
plt.show()


## Compare the Feature Choice

The groups can change because we changed the information given to the algorithm.

That is not a mistake.

We asked a different question.


In [ ]:
age_spend_profile = customers.groupby(
    "Age_Spend_Cluster"
)[["Age", "Spending_Score"]].mean().round(2)

age_spend_profile


## Example 2: Age + Income

Now imagine the business wants to understand customer demographics rather than spending behaviour.

We can cluster using age and income.


In [ ]:
X_age_income = customers[
    ["Age", "Income"]
]

age_income_scaler = StandardScaler()

X_age_income_scaled = age_income_scaler.fit_transform(
    X_age_income
)

age_income_model = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

customers["Age_Income_Cluster"] = age_income_model.fit_predict(
    X_age_income_scaled
)

customers.head()


In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    customers["Age"],
    customers["Income"],
    c=customers["Age_Income_Cluster"]
)

plt.xlabel("Age")
plt.ylabel("Income")
plt.title("Age and Income Clusters")
plt.show()


## Why Do We Not Just Use Every Column?

More features are not automatically better.

Adding a feature changes the distances used by K-Means.

We should choose features based on the problem we are trying to solve and the quality of the available data.


## What If We Want to Use Three Features?

We can use:

- Age
- Income
- Spending Score

This gives the model more information about each customer.


In [ ]:
X_three = customers[
    ["Age", "Income", "Spending_Score"]
]

three_scaler = StandardScaler()

X_three_scaled = three_scaler.fit_transform(X_three)

three_model = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

customers["Three_Feature_Cluster"] = three_model.fit_predict(
    X_three_scaled
)

customers.head()


## Why Do We Not Use a Normal 2D Scatter Plot for Three Features?

A standard scatter plot has:

- one x-axis
- one y-axis

So it naturally shows two variables.

We can use 3D visualization, but it is often harder to interpret than a simple 2D plot.

For beginners, we keep the two-feature model as our main visual example.


In [ ]:
fig = plt.figure(figsize=(9, 7))

ax = fig.add_subplot(111, projection="3d")

ax.scatter(
    customers["Age"],
    customers["Income"],
    customers["Spending_Score"],
    c=customers["Three_Feature_Cluster"]
)

ax.set_xlabel("Age")
ax.set_ylabel("Income")
ax.set_zlabel("Spending Score")
ax.set_title("Three-Feature Customer Clustering")

plt.show()


# Part 6 — Other Clustering Approaches

K-Means is not the only clustering algorithm.

Other common approaches include:

- Hierarchical Clustering
- DBSCAN
- Gaussian Mixture Models

We will not implement all of them today because the curriculum specifically introduces **K-Means** for Week 4 Day 2.


## Why K-Means for This Lesson?

K-Means is a good first clustering algorithm because:

- The idea is relatively easy to visualize.
- The steps are straightforward.
- It works naturally with numerical features.
- The centroid idea gives beginners an intuitive way to understand clustering.
- It directly satisfies our Week 4 clustering objective.


## When K-Means May Not Be the Best Choice

K-Means may struggle when:

- The clusters have unusual shapes.
- There are many strong outliers.
- The correct number of clusters is unclear.
- Features are poorly scaled.
- The data does not naturally form centroid-based groups.


## Alternative: Hierarchical Clustering

Hierarchical clustering builds a hierarchy of groups.

It can be useful when we want to examine how clusters merge or split at different levels.

We are not using it as the main algorithm today because our curriculum focuses on K-Means.


## Alternative: DBSCAN

DBSCAN groups points based on density and can identify some unusual observations as noise.

It can be useful when clusters are irregularly shaped.

Again, we keep it as an alternative concept rather than replacing K-Means in this lesson.


# Part 7 — Comparing Different Values of K

Let's directly compare `K=2`, `K=3`, and `K=4`.

This helps students see why choosing K is a modeling decision.


In [ ]:
for k in [2, 3, 4]:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X_scaled)

    plt.figure(figsize=(7, 5))

    plt.scatter(
        customers["Income"],
        customers["Spending_Score"],
        c=labels
    )

    plt.xlabel("Income")
    plt.ylabel("Spending Score")
    plt.title(f"Customer Clustering with K = {k}")
    plt.show()


## What Should You Observe?

As `K` changes, the algorithm creates a different segmentation.

There is no universal rule saying that one value of K is always correct.

We combine:

- Elbow Method
- Silhouette Score
- Visual inspection
- Business understanding


# Part 8 — Understanding Important K-Means Settings

You have already seen:

```python
KMeans(n_clusters=3, random_state=42, n_init=10)
```

Let's understand why each setting is there.


## `n_clusters`

`n_clusters` tells K-Means how many groups to create.

Example:

```python
KMeans(n_clusters=3)
```

means:

> Create three clusters.


In [ ]:
example_model = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

print(example_model)


## `random_state`

K-Means can use randomness when choosing starting centroids.

`random_state=42` makes the experiment reproducible.

The number 42 is not special. Another fixed integer can also be used.


## `n_init`

K-Means can be started from different initial centroid positions.

`n_init=10` tells Scikit-learn to run multiple initializations and keep the better result according to the model's objective.

This helps reduce dependence on one unlucky initialization.


## Why We Use These Settings

For a beginner notebook, this gives us:

- A clear number of clusters.
- Reproducible results.
- More reliable initialization than relying on a single starting point.


# Part 9 — Example 2: Grouping Students

K-Means is not only for customers.

Imagine a school wants to discover student behaviour patterns using:

- Study Hours
- Attendance


In [ ]:
students = pd.DataFrame({
    "Student": ["A", "B", "C", "D", "E", "F", "G", "H", "I"],
    "Study_Hours": [2, 3, 2.5, 7, 8, 7.5, 12, 11, 13],
    "Attendance": [65, 70, 68, 82, 85, 80, 95, 92, 97]
})

students


## Visualize the Students

Before clustering, look at the data.


In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    students["Study_Hours"],
    students["Attendance"]
)

plt.xlabel("Study Hours")
plt.ylabel("Attendance")
plt.title("Student Behaviour Before Clustering")
plt.show()


## Select and Scale the Features

We use both variables because the question is:

> Can we discover groups of students with similar study and attendance behaviour?


In [ ]:
student_X = students[
    ["Study_Hours", "Attendance"]
]

student_scaler = StandardScaler()

student_X_scaled = student_scaler.fit_transform(
    student_X
)

student_X_scaled


## Train K-Means With 3 Clusters


In [ ]:
student_model = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

students["Cluster"] = student_model.fit_predict(
    student_X_scaled
)

students


## Visualize Student Clusters


In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    students["Study_Hours"],
    students["Attendance"],
    c=students["Cluster"]
)

plt.xlabel("Study Hours")
plt.ylabel("Attendance")
plt.title("Student Behaviour Clusters")
plt.show()


## Profile the Student Clusters


In [ ]:
students.groupby("Cluster")[
    ["Study_Hours", "Attendance"]
].mean().round(2)


## Interpret Carefully

A cluster might contain students with:

- Lower study hours and lower attendance
- Moderate study hours and attendance
- Higher study hours and attendance

These are descriptions of the observed data.

They are not diagnoses or judgments about the students.


# Part 10 — Example 3: Product Clustering

A company may also want to group products based on:

- Price
- Monthly Sales


In [ ]:
products = pd.DataFrame({
    "Product": list("ABCDEFGHI"),
    "Price": [20, 25, 22, 80, 90, 85, 45, 50, 48],
    "Monthly_Sales": [900, 850, 920, 300, 250, 280, 600, 550, 620]
})

products


## Visualize Products


In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    products["Price"],
    products["Monthly_Sales"]
)

plt.xlabel("Price")
plt.ylabel("Monthly Sales")
plt.title("Products Before Clustering")
plt.show()


## Scale Product Features


In [ ]:
product_X = products[
    ["Price", "Monthly_Sales"]
]

product_scaler = StandardScaler()

product_X_scaled = product_scaler.fit_transform(
    product_X
)


## Apply K-Means


In [ ]:
product_model = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

products["Cluster"] = product_model.fit_predict(
    product_X_scaled
)

products


## Visualize Product Clusters


In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    products["Price"],
    products["Monthly_Sales"],
    c=products["Cluster"]
)

plt.xlabel("Price")
plt.ylabel("Monthly Sales")
plt.title("Product Clusters")
plt.show()


## Profile the Products


In [ ]:
products.groupby("Cluster")[
    ["Price", "Monthly_Sales"]
].mean().round(2)


## What Did We Learn?

The exact algorithm stayed the same.

Only the **problem and features changed**.

This is a major Machine Learning idea:

> The same algorithm can be applied to different real-world problems when the data and question are different.


# Part 11 — Evaluating Clustering Results

In supervised learning, we compared predictions with known answers.

Clustering is different because there is usually no known target telling us the "correct" cluster.

So evaluation is based on measures such as:

- Inertia
- Silhouette Score
- Visual structure
- Domain knowledge


## Why Accuracy Is Not Used Here

Accuracy asks:

> How many predictions were correct?

But in our clustering problem, we do not have a column saying:

`Correct_Cluster = 1`

Therefore, ordinary prediction accuracy is not the right evaluation approach for this unsupervised task.


In [ ]:
customer_silhouette = silhouette_score(
    X_scaled,
    customers["Cluster"]
)

print(
    "Customer Silhouette Score:",
    round(customer_silhouette, 3)
)


## Evaluate the Student Example


In [ ]:
student_silhouette = silhouette_score(
    student_X_scaled,
    students["Cluster"]
)

print(
    "Student Silhouette Score:",
    round(student_silhouette, 3)
)


## Evaluate the Product Example


In [ ]:
product_silhouette = silhouette_score(
    product_X_scaled,
    products["Cluster"]
)

print(
    "Product Silhouette Score:",
    round(product_silhouette, 3)
)


## Interpreting Evaluation

A stronger silhouette score generally suggests that the clusters are:

- relatively cohesive within themselves
- relatively separated from other clusters

But the score does not tell us whether the segmentation is useful to a business.

That still requires interpretation.


## K-Means Limitation: Outliers

Imagine one customer has an extremely unusual income.

Because K-Means uses centroids, an extreme observation can influence the position of a cluster center.

This is why data exploration and cleaning still matter before modeling.


In [ ]:
outlier_example = pd.DataFrame({
    "Income": [20, 22, 24, 25, 27, 29, 31, 200],
    "Spending_Score": [50, 52, 48, 55, 51, 54, 53, 95]
})

plt.figure(figsize=(8, 5))

plt.scatter(
    outlier_example["Income"],
    outlier_example["Spending_Score"]
)

plt.xlabel("Income")
plt.ylabel("Spending Score")
plt.title("Example of an Extreme Observation")
plt.show()


## K-Means Limitation: Choosing K

K-Means requires us to choose K.

The model does not automatically know that a business wants exactly three customer groups.

That is why we introduced the Elbow Method and Silhouette Score.


## K-Means Limitation: Cluster Shape

K-Means is based on distance to centroids.

If the real groups have unusual shapes, K-Means may not represent them well.

This is one reason other clustering algorithms exist.


# Part 12 — Guided Practice

Now it is your turn.

The exercises move from simple application to interpretation.


### Exercise 1 — Beginner

**Objective:** Load `customers.csv` and display:

- the first five rows
- dataset information
- summary statistics

**Estimated Time:** 5 minutes


In [ ]:
# Write your code here


### Exercise 2 — Basic K-Means

**Objective:** Use `Income` and `Spending_Score`, then apply K-Means with `K=3`.

**Estimated Time:** 8 minutes

**Hint:** Use `StandardScaler` before training.


In [ ]:
# Write your code here


### Exercise 3 — Visualization

**Objective:** Plot the three clusters using:

- Income on the x-axis
- Spending Score on the y-axis

Add a title and axis labels.

**Estimated Time:** 7 minutes


In [ ]:
# Write your code here


### Exercise 4 — Cluster Profile

**Objective:** Use `groupby()` to calculate the average:

- Age
- Income
- Spending Score

for each cluster.

**Estimated Time:** 7 minutes


In [ ]:
# Write your code here


### Exercise 5 — Choose K

**Objective:** Test K values from 2 to 6.

Calculate:

- inertia
- silhouette score

Then decide which K you would investigate further.

**Estimated Time:** 12 minutes


In [ ]:
# Write your code here


### Exercise 6 — Change the Features

**Objective:** Repeat the clustering using:

`Age + Spending_Score`

Compare the resulting groups with your first model.

**Estimated Time:** 10 minutes


In [ ]:
# Write your code here


### Exercise 7 — Business Interpretation

**Objective:** Look at your final cluster profile.

For each cluster, write:

1. A meaningful cluster name.
2. Two characteristics.
3. One possible business action.

**Estimated Time:** 10 minutes


In [ ]:
# Write your interpretation here


# Challenge Activity — Customer Segmentation

You are working as a junior data scientist for a retail company.

The company says:

> "We have customer data, but we do not know what types of customers we have. Help us discover useful customer groups."


## Your Task

Build a complete K-Means workflow:

1. Load `customers.csv`.
2. Inspect the data.
3. Choose relevant features.
4. Explain why you selected them.
5. Scale the features.
6. Test different values of K.
7. Use the Elbow Method.
8. Use the Silhouette Score.
9. Train a final K-Means model with at least 3 clusters.
10. Visualize the clusters.
11. Create a cluster profile.
12. Give each cluster a meaningful interpretation.
13. Recommend one possible action for each cluster.


In [ ]:
# Challenge solution

# 1. Load data
# 2. Select features
# 3. Scale features
# 4. Choose K
# 5. Train K-Means
# 6. Add cluster labels
# 7. Visualize
# 8. Profile and interpret


# Debug This Code

The code below is intended to cluster customers.

There are mistakes.

Find and fix them.

This is important because real Machine Learning work is not only writing code; it is also reading, debugging, and improving code.


In [ ]:
# BUGGY CODE

# X = customers[["Income", "Spending Score"]]

# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)

# model = KMeans(n_clusters=3 random_state=42)

# labels = model.fit_predict(X_scaled)

# customers["Cluster"] = labels

# customers.head()


### Debugging Hint

Look carefully at:

- column names
- commas
- parentheses
- the K-Means arguments

Do not rewrite everything. Fix the existing code.


In [ ]:
# Write your corrected version here


# Functions and Concepts Learned Today

Keep this table at the end of your notebook as a quick reference.


| Item | Purpose |
|---|---|
| `KMeans()` | Creates a K-Means clustering model |
| `n_clusters` | Sets the number of clusters |
| `random_state` | Makes the experiment reproducible |
| `n_init` | Runs multiple centroid initializations |
| `.fit()` | Learns the cluster structure |
| `.fit_predict()` | Learns clusters and returns labels |
| `cluster_centers_` | Returns the learned centroid positions |
| `inertia_` | Measures within-cluster squared distance |
| `StandardScaler()` | Standardizes numerical features |
| `.fit_transform()` | Learns scaling information and transforms data |
| `inverse_transform()` | Converts scaled values back toward the original scale |
| `silhouette_score()` | Evaluates cluster cohesion and separation |
| `groupby()` | Summarizes observations by cluster |
| `plt.scatter()` | Displays relationships between two numerical variables |
| `plt.plot()` | Creates a line chart |
| `plt.xlabel()` | Labels the x-axis |
| `plt.ylabel()` | Labels the y-axis |
| `plt.title()` | Adds a chart title |
| `plt.show()` | Displays the chart |


## Choosing the Right Step

| Situation | What We Use | Why |
|---|---|---|
| Need to inspect data | Pandas | Understand the dataset first |
| Need to compare two numerical features | Scatter plot | Easy visual inspection |
| Features have different scales | `StandardScaler` | Makes distance calculations more balanced |
| Need customer groups | `KMeans` | Direct clustering algorithm for this lesson |
| Need to compare possible K values | Elbow Method | Shows change in inertia |
| Need cluster separation information | Silhouette Score | Measures cohesion and separation |
| Need to understand each cluster | `groupby()` | Creates a readable cluster profile |


# Lesson Summary

Today, you learned how to:

- Explain unsupervised learning.
- Explain clustering.
- Explain K-Means.
- Understand K and centroids.
- Understand why distance matters.
- Select relevant features.
- Scale numerical features.
- Apply K-Means with 3 clusters.
- Visualize clusters.
- Inspect cluster centroids.
- Profile clusters.
- Use the Elbow Method.
- Use the Silhouette Score.
- Compare different K values.
- Compare different feature choices.
- Interpret clusters in real-world scenarios.
- Recognize important limitations of K-Means.


# Key Takeaways

✅ **Unsupervised Learning** discovers patterns without a known target.

✅ **Clustering** groups similar observations.

✅ **K-Means** divides data into a chosen number of clusters.

✅ **K** is the number of clusters we ask the model to create.

✅ **Centroids** represent the centers of clusters.

✅ Because K-Means uses distance, **feature scaling matters**.

✅ The **Elbow Method** and **Silhouette Score** help us investigate a suitable K.

✅ Cluster numbers such as `0`, `1`, and `2` have no inherent meaning.

✅ We interpret clusters using their actual characteristics.

✅ A useful Machine Learning result requires both **technical evaluation and real-world interpretation**.


# Assignment

Submit a Jupyter Notebook containing:

1. Dataset description.
2. Customer dataset loaded with Pandas.
3. Selected features and an explanation of why they were selected.
4. Feature scaling.
5. K-Means implementation with at least 3 clusters.
6. Cluster visualization using a scatter plot.
7. Elbow Method.
8. Silhouette Score.
9. Cluster profile table.
10. Meaningful interpretation of each cluster.
11. One possible business action for each cluster.
12. A short paragraph explaining the limitations of your clustering result.


# Final Reflection

Answer these questions in your notebook:

1. Why is K-Means called unsupervised learning?
2. Why does K-Means need a value for K?
3. Why did we scale the features?
4. Why can we not say `Cluster 0` automatically means "low spender"?
5. What did the Elbow Method help us investigate?
6. What does the Silhouette Score tell us?
7. How did the results change when we changed the features?
8. What would you do differently if you had a much larger real-world customer dataset?


# Preview: Week 5

Next, we move from **grouping** data to **classification**.

Instead of asking:

> "What groups naturally exist?"

we will ask:

> "Which category does this new observation belong to?"

We will begin with **Logistic Regression**.


# Quick Knowledge Check

Before leaving this notebook, explain this pipeline in your own words:

**Data → Features → Scaling → K → K-Means → Clusters → Visualization → Evaluation → Interpretation**


## One Final Real-World Reminder

A clustering model does not make the business decision for us.

It gives us a structured way to discover patterns.

The data scientist's job is to determine whether those patterns are meaningful, reliable, and useful.
